# Build Multi-Agent AI System

This project builds a multi-agent AI system using **CrewAI** where specialized agents collaborate to research a topic and produce written content — all automated end-to-end.

**Workflow:**
```
Researcher Agent
     |
     | (research findings passed as context)
     ↓
Writer Agent  ──(if needs more info)──→  delegates back to Researcher
     |
     ↓
 Final Blog Post
```

> **Note:** The writer doesn't always go back to the researcher — it only delegates if it decides more information is needed (`allow_delegation=True`). In a `Process.sequential` setup, tasks run in order and the output of each task is automatically passed as context to the next one.

In [ ]:
%%capture

from crewai import LLM, Agent, Task, Crew, Process
from crewai_tools import SerperDevTool
from crewai import 

from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

`Add Serper and OpenAI API key in env with keys SERPER_API_KEY and OPENAI_API_KEY`

In [ ]:
# search_tool=SerperDevTool(n_results=5, save_file=True, search_type="search")
search_tool=SerperDevTool()
print(type(search_tool))

<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool'>


In [5]:
search_query = "Latest Breakthroughs in machine learning"
search_results =search_tool.run(query=search_query)

Using Tool: Search the internet


In [ ]:
records = search_results.split("---")
print(f"Total Number of records: {len(records)}")

Total records: 11


In [ ]:
print(search_results[:1000])

Total No. of search results: 2815

Search results: Title: Advancements in AI and Machine Learning
Link: https://ep.jhu.edu/news/advancements-in-ai-and-machine-learning/
Snippet: AI and ML advancements are transforming engineering by automating complex tasks and enhancing decision-making processes for professionals.
---
Title: 5 Breakthrough Machine Learning Research Papers Already in 2025
Link: https://machinelearningmastery.com/5-breakthrough-machine-learning-research-papers-already-in-2025/
Snippet: 1. SAM 2: Segment Anything in Images and Videos · 2. Learning Dynamics of LLM Finetuning · 3. Data Shapley in One Training Run · 4. Faster Cascades ...
---
Title: Latest AI News and AI Breakthroughs that Matter Most: 2026
Link: https://www.crescendo.ai/news/latest-ai-news-and-updates
Snippet: Wondering what's happening in the AI world? Here are the latest AI breakthroughs and news that are shaping the world around us!
---
Title: Machine learning - Latest research and news | Nature
Link: h

In [2]:
llm = LLM(
    model="openai/gpt-4o-mini",
    max_tokens=2000,
)

## Agents in CrewAI

A CrewAI agent isn’t just a block of code; it’s a thoughtfully designed entity with the following parameters:  

1. **Role**  
   An agent’s role defines its purpose in the system. Roles are as diverse as your project needs, such as a **"Data Researcher"** hunting for insights or a **"Reporting Analyst"** preparing comprehensive summaries.  

2. **Goal**  
   Each agent operates with a defined goal—a guiding star that shapes its decisions and actions. For instance, an agent with the goal to **“Uncover cutting-edge developments in AI”** will consistently align its behavior to fulfill this objective.  

3. **Backstory**  
   An agent’s backstory is like its resume, providing context or personality that influences how it behaves and interacts. For example, a seasoned **“Senior Data Researcher”** with years of experience might approach tasks differently from a **“Junior Analyst”** just starting out. This feature adds depth and relatability to agent interactions, making them more dynamic and tailored.  


4. **Tools**  
   Just like any professional needs the right tools to excel, agents in CrewAI are equipped with specialized tools to boost their performance. Whether it’s a **web search utility** for gathering information, a **data analysis engine** for crunching numbers, or an **API connector** to integrate external services, tools expand an agent’s capabilities. The right tool can help an agent complete its tasks more efficiently and effectively, enabling it to work smarter, not harder.  

5. **Configuration**  
   Agents in CrewAI are configured using simple YAML files, offering a modular, readable, and scalable approach to defining their attributes. This makes setting up agents intuitive, even for large systems ( in this tutroal we will not use a YML files 




In [ ]:
# Define the researcher Agent
research_agent = Agent(
  role='Senior Research Analyst',
  goal='Uncover cutting-edge information and insights on any subject with comprehensive analysis',
  backstory="""You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
  Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
  You excel at finding reliable sources and extracting valuable information efficiently.""",
  verbose=True, # for logging, agent level thinking
  allow_delegation=False,
  llm=llm,
  tools=[SerperDevTool()]
)

In [4]:
research_agent

Agent(role=Senior Research Analyst, goal=Uncover cutting-edge information and insights on any subject with comprehensive analysis, backstory=You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
  Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
  You excel at finding reliable sources and extracting valuable information efficiently.)

NOTE: `We defined agent directly as python code. Defining it via yaml file is preferred approach.`

In [ ]:
# Define the Writer Agent
writer_agent = Agent(
  role='Tech Content Strategist',
  goal='Craft well-structured and engaging content based on research findings',
  backstory="""You are a skilled content strategist known for translating 
  complex topics into clear and compelling narratives. Your writing makes 
  information accessible and engaging for a wide audience.""",
  # verbose=True, # for logging, agent level thinking
  llm = llm,
  allow_delegation=True, # Enables task assignment to other agents
)

The agents work together in a coordinated flow:

1. **Researcher** does initial research → passes findings to **Writer**
2. **Writer** reviews, and if it needs more detail, it delegates back → *"research more on X"*
3. **Researcher** does the extra research → returns to **Writer**
4. **Writer** completes the final blog post

That's why `allow_delegation=True` is only on the writer — it drives the quality of the final output and can pull in more research as needed, while the researcher just executes whatever it's asked.

In [6]:
writer_agent

Agent(role=Tech Content Strategist, goal=Craft well-structured and engaging content based on research findings, backstory=You are a skilled content strategist known for translating 
  complex topics into clear and compelling narratives. Your writing makes 
  information accessible and engaging for a wide audience.)

## Tasks in CrewAI

Tasks are like to-do items for our AI agents. Each task has specific instructions, details, and tools for the agent to follow and complete the job.

For example:
- A task could ask an agent to "research the latest AI trends."
- Another task could ask a different agent to "write a detailed report based on the research."

#### **How Tasks Work**  
There are two ways tasks can run:  

1. **Sequential**: Tasks are executed one after the other, like following a recipe step-by-step. Each task waits for the previous one to finish.  
2. **Hierarchical**: Tasks are assigned based on agent skills or roles, and multiple tasks can run in parallel if they don’t depend on each other.  




In [ ]:
# Create a task for the Researcher Agent
research_task = Task(
  description="Analyze the major {topic}, identifying key trends and technologies. Provide a detailed report on their potential impact.",
  agent=research_agent,
  expected_output="A detailed report on {topic}, including trends, emerging technologies, and their impact."
)

In [8]:
# Create a task for the Writer Agent
writer_task = Task(
  description="Create an engaging blog post based on the research findings about {topic}. Tailor the content for a tech-savvy audience, ensuring clarity and interest.",
  agent=writer_agent,
  expected_output="A 4-paragraph blog post on {topic}, written clearly and engagingly for tech enthusiasts."
)

## CrewAI Workflow

In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent],
    tasks=[research_task, writer_task],
    process=Process.sequential, # tasks will run one after another in the specified order (research first, then writing)
    verbose=True 
)

# To reduce log output:
# - Set verbose=False on agents (research_agent, writer_agent) to silence per-agent thought logs
# - Set verbose=False on Crew to silence all crew-level logs
# - Keep verbose=True only on Crew for compact status lines without agent thought dumps

In [ ]:
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs"})

In [12]:
type(result)

crewai.crews.crew_output.CrewOutput

In [11]:
result

CrewOutput(raw="### The Latest Breakthroughs in Generative AI: A Game Changer for 2023\n\n2023 is a pivotal year in the realm of generative AI, showcasing an extraordinary convergence of technological advancements that are reshaping various industries. According to a recent McKinsey Global Survey, approximately one-third of organizations now leverage generative AI tools regularly, marking a significant milestone in the technology's adoption. This rapid integration highlights the transformative power of generative AI, enabling businesses to redefine their operational efficiencies and innovate across diverse sectors.\n\nAmong the notable breakthroughs, multimodal AI models stand out, particularly OpenAI's GPT-4. This model sets a new precedent in processing and generating content across multiple formats—text, images, and even audio. Such enhancements have dramatically enriched user interactions by providing contextually relevant responses tailored to specific platforms. Coupled with the 

In [13]:
final_output = result.raw
print("Final output:", final_output)

Final output: ### The Latest Breakthroughs in Generative AI: A Game Changer for 2023

2023 is a pivotal year in the realm of generative AI, showcasing an extraordinary convergence of technological advancements that are reshaping various industries. According to a recent McKinsey Global Survey, approximately one-third of organizations now leverage generative AI tools regularly, marking a significant milestone in the technology's adoption. This rapid integration highlights the transformative power of generative AI, enabling businesses to redefine their operational efficiencies and innovate across diverse sectors.

Among the notable breakthroughs, multimodal AI models stand out, particularly OpenAI's GPT-4. This model sets a new precedent in processing and generating content across multiple formats—text, images, and even audio. Such enhancements have dramatically enriched user interactions by providing contextually relevant responses tailored to specific platforms. Coupled with the rise o

The **tasks_output** list gives us access to outputs from each task in the order they were executed:

In [ ]:
tasks_outputs = result.tasks_output

In [ ]:
tasks_outputs[0]

TaskOutput(description='Analyze the major Latest Generative AI breakthroughs, identifying key trends and technologies. Provide a detailed report on their potential impact.', name='Analyze the major Latest Generative AI breakthroughs, identifying key trends and technologies. Provide a detailed report on their potential impact.', expected_output='A detailed report on Latest Generative AI breakthroughs, including trends, emerging technologies, and their impact.', summary='Analyze the major Latest Generative AI breakthroughs, identifying key trends...', raw="### Detailed Report on Latest Generative AI Breakthroughs (2023)\n\n#### 1. Overview of the Current Landscape\n2023 has marked a significant turning point for generative AI, driven by rapid advancements and increasing integration into various sectors. The McKinsey Global Survey reported that one-third of organizations are regularly employing generative AI tools, a testament to its burgeoning adoption across industries.\n\n#### 2. Key B

In [ ]:
# Available keys in tasks outputs
print(list(tasks_outputs[0].model_fields.keys()))

['description', 'name', 'expected_output', 'summary', 'raw', 'pydantic', 'json_dict', 'agent', 'output_format', 'messages']


In [15]:
print("Researcher Task Description", tasks_outputs[0].description)
print("Output of research task ",tasks_outputs[0].raw)

Researcher Task Description Analyze the major Latest Generative AI breakthroughs, identifying key trends and technologies. Provide a detailed report on their potential impact.
Output of research task  ### Detailed Report on Latest Generative AI Breakthroughs (2023)

#### 1. Overview of the Current Landscape
2023 has marked a significant turning point for generative AI, driven by rapid advancements and increasing integration into various sectors. The McKinsey Global Survey reported that one-third of organizations are regularly employing generative AI tools, a testament to its burgeoning adoption across industries.

#### 2. Key Breakthroughs
- **Multimodal AI Models**: Enhancements in AI capabilities to process and generate content across various formats (text, images, audio) have become noteworthy. For instance, OpenAI's GPT-4 model has been pivotal, drastically improving user experience by offering contextually relevant responses across platforms ([Stanford HAI](https://hai.stanford.ed

In [16]:
print("Writer task description:", tasks_outputs[1].description)
print(" \nOutput of writer task:", tasks_outputs[1].raw)

Writer task description: Create an engaging blog post based on the research findings about Latest Generative AI breakthroughs. Tailor the content for a tech-savvy audience, ensuring clarity and interest.
 
Output of writer task: ### The Latest Breakthroughs in Generative AI: A Game Changer for 2023

2023 is a pivotal year in the realm of generative AI, showcasing an extraordinary convergence of technological advancements that are reshaping various industries. According to a recent McKinsey Global Survey, approximately one-third of organizations now leverage generative AI tools regularly, marking a significant milestone in the technology's adoption. This rapid integration highlights the transformative power of generative AI, enabling businesses to redefine their operational efficiencies and innovate across diverse sectors.

Among the notable breakthroughs, multimodal AI models stand out, particularly OpenAI's GPT-4. This model sets a new precedent in processing and generating content ac

In [17]:
print("We can get the agent for researcher task:  ",tasks_outputs[0].agent)
print("We can get the agent for the writer task: ",tasks_outputs[1].agent)

We can get the agent for researcher task:   Senior Research Analyst
We can get the agent for the writer task:  Tech Content Strategist


CrewAI provides detailed performance metrics that help you monitor resource usage and optimize your multi-agent systems


In [24]:
token_count = result.token_usage.total_tokens
prompt_tokens = result.token_usage.prompt_tokens
completion_tokens = result.token_usage.completion_tokens

print(f"Total tokens used: {token_count}")
print(f"Prompt tokens: {prompt_tokens} (used for instructions to the model)")
print(f"Completion tokens: {completion_tokens} (generated in response)")

Total tokens used: 50700
Prompt tokens: 47762 (used for instructions to the model)
Completion tokens: 2938 (generated in response)


---

#### Create a Social Media Strategist Agent

In [26]:
# Define the Social Media Strategist Agent
strategist_agent = Agent(
  role='Social Media Strategist',
  goal='Create a well-crafted and short-version post for linked and twitter(X) from research findings',
  backstory="""You are a skilled content writer for Linked and Twitter(X). You take research findings and create a short and effective version of posts that will attract tech-enthusiasts.""",
  # verbose=True, # for logging, agent level thinking
  llm = llm,
  allow_delegation=False, # Enables task assignment to other agents
)

# social_agent = Agent(
#     role='Social Media Strategist',
#     goal='Generate engaging social media snippets based on the full article',
#     backstory="A digital storyteller who excels at crafting compelling posts to drive engagement and traffic.",
#     verbose=True
# )

#### Defining a Social Media Strategy Task

In [27]:
# Create a task for the Writer Agent
strategist_task = Task(
  description="Create an engaging post based on the research findings about {topic}. Tailor the content for a tech-savvy audience, ensuring clarity and interest.",
  agent=strategist_agent,
  expected_output="A short summarized version of post on {topic}, written clearly and engagingly for tech enthusiasts."
)


In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent, strategist_agent],
    tasks=[research_task, writer_task, strategist_task],
    process=Process.sequential, # Tasks will be executed one after another
    verbose=True 
)

In [ ]:
# Run the crew and capture the final output (includes research, blog post, and social media content)
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs in 2026"})